In [1]:
%pip install plotly nbformat

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Note: you may need to restart the kernel to use updated packages.


In [10]:
import pandas as pd

# Parsed data from sl_results.txt (supervised)
sl_data = {
    'Method': [],
    'Coverage Mean': [],
    'Coverage Std': [],
    'Coverage Min': [],
    'Coverage Max': [],
    'Ratio Mean': [],
    'Ratio Std': [],
    'Ratio Min': [],
    'Ratio Max': [],
}

# Add supervised learning results for each size category
sl_categories = [
    ("10-59", 0.95, 0.08, None, None, 6.26, 2.43, None, None),
    ("60-99", 0.93, 0.06, None, None, 11.75, 2.53, None, None),
    ("100-199", 0.87, 0.07, None, None, 19.89, 4.19, None, None)
   
]

for size_range, c_mean, c_std, c_min, c_max, r_mean, r_std, r_min, r_max in sl_categories:
    sl_data['Method'].append(f"Supervised {size_range}")
    sl_data['Coverage Mean'].append(c_mean)
    sl_data['Coverage Std'].append(c_std)
    sl_data['Coverage Min'].append(c_mean - c_std * 2)
    sl_data['Coverage Max'].append(c_mean + c_std * 2)
    sl_data['Ratio Mean'].append(r_mean)
    sl_data['Ratio Std'].append(r_std)
    sl_data['Ratio Min'].append(max(0, r_mean - r_std * 2))
    sl_data['Ratio Max'].append(r_mean + r_std * 2)

# Aggregate supervised results
import numpy as np
sl_agg = {
    'Method': 'Supervised Learning',
    'Coverage Mean': np.mean([x[1] for x in sl_categories]),
    'Coverage Std': np.mean([x[2] for x in sl_categories]),
    'Coverage Min': np.mean([x[1] - x[2]*2 for x in sl_categories]),
    'Coverage Max': np.mean([x[1] + x[2]*2 for x in sl_categories]),
    'Ratio Mean': np.mean([x[5] for x in sl_categories]),
    'Ratio Std': np.mean([x[6] for x in sl_categories]),
    'Ratio Min': np.mean([max(0, x[5] - x[6]*2) for x in sl_categories]),
    'Ratio Max': np.mean([x[5] + x[6]*2 for x in sl_categories]),
}

# Parsed data from comparison.txt (RL and Greedy)
comparison_data = {
    'Method': ['Reinforcement Learning', 'Greedy'],
    'Coverage Mean': [0.9057, 1.0000],
    'Coverage Std': [0.1992, 0.0000],
    'Coverage Min': [0.0016, 1.0000],
    'Coverage Max': [1.0000, 1.0000],
    'Ratio Mean': [3.4913, 1.1721],
    'Ratio Std': [2.9660, 0.1239],
    'Ratio Min': [0.0303, 1.0000],
    'Ratio Max': [55.0000, 2.0000],
}

# Create DataFrames
sl_df = pd.DataFrame(sl_data)
sl_agg_df = pd.DataFrame([sl_agg])
comparison_df = pd.DataFrame(comparison_data)

# Merge into a unified table (RL, Greedy, Supervised Aggregate only)
unified_df = pd.concat([comparison_df, sl_agg_df], ignore_index=True)

# Rearrange rows: Greedy, Supervised Learning, Reinforcement Learning
order = ['Greedy', 'Supervised Learning', 'Reinforcement Learning']
unified_df = unified_df.set_index('Method').loc[order].reset_index()

In [11]:
unified_df.head()

,Method,Coverage Mean,Coverage Std,Coverage Min,Coverage Max,Ratio Mean,Ratio Std,Ratio Min,Ratio Max
0,Greedy,1.000000,0.0000,1.000000,1.000000,1.172100,0.1239,1.000000,2.000000
1,Supervised Learning,0.916667,0.0700,0.776667,1.056667,12.633333,3.0500,6.533333,18.733333
2,Reinforcement Learning,0.905700,0.1992,0.001600,1.000000,3.491300,2.9660,0.030300,55.000000


In [13]:
import plotly.graph_objects as go

fig = go.Figure()

# Coverage bar
fig.add_trace(go.Bar(
    x=unified_df['Method'],
    y=unified_df['Coverage Mean'],
    name='Coverage',
    error_y=dict(
        type='data',
        array=unified_df['Coverage Std'],
        visible=True
    ),
    marker_color='royalblue',
    offsetgroup=0,
))

# Ratio bar
fig.add_trace(go.Bar(
    x=unified_df['Method'],
    y=unified_df['Ratio Mean'],
    name='Ratio',
    error_y=dict(
        type='data',
        array=unified_df['Ratio Std'],
        visible=True
    ),
    marker_color='orange',
    offsetgroup=1,
))

fig.update_layout(
    title='Coverage and Ratio Comparison',
    xaxis=dict(title='Method'),
    yaxis=dict(title='Value'),
    barmode='group',
    legend=dict(x=0.5, y=1.1, orientation='h')
)

fig.show()

In [ ]:
%pip install nbformat